# 🎵 AICoverGen WebUI — Kaggle versiyasi (O'zbekcha)
Ushbu notebook **AICoverGen** dasturini **Kaggle** platformasida bepul GPU (T4 yoki P100) yordamida xatosiz ishga tushirish uchun maxsus moslashtirilgan.

---
### ⚠️ ISHGA TUSHIRISH OLDIDAN ENG MUHIM SOZLAMALAR (O'ng tarafdagi menyu):
1. **Internet**: `Settings` -> `Internet` -> **ON (Yoqilgan)** bo'lishi SHART! *(Agar yoqilmasa, telefon raqamingiz orqali Kaggle hisobingizni tasdiqlang - bepul)*
2. **Accelerator**: `Settings` -> `Accelerator` -> **GPU T4 x2** yoki **GPU P100** ni tanlang.

---
### 🚀 Foydalanish tartibi:
- **1-QADAM:** Tizim kutubxonalari, modellar va RVC fayllarini o'rnatish (~3-4 daqiqa).
- **2-QADAM:** WebUI ni ishga tushirish va berilgan havola (Cloudflare, Gradio yoki Ngrok) orqali kirish!

In [ ]:
# ==========================================================
# 📦 1-QADAM: Kerakli muhit, kutubxonalar va AI modellarni o'rnatish
# ==========================================================
import os, sys, shutil, re, subprocess
from IPython.display import clear_output

print("🔄 1/5 Tizim paketlari o'rnatilmoqda (sox, ffmpeg, cloudflared)...")
!apt-get update -y -qq > /dev/null
!apt-get install -y -qq sox libsox-dev libsox-fmt-all ffmpeg curl wget > /dev/null

# Cloudflared tunnelni o'rnatish (tokensiz, bepul va qulay tashqi havola uchun)
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

print("🔄 2/5 Pip va mos kutubxonalar (Numpy 1.26, Hydra, OmegaConf) sozlanmoqda...")
!pip install -q --no-warn-conflicts "pip<24.1"
!pip install -q --no-warn-conflicts cython "numpy==1.26.4" hydra-core==1.0.7 omegaconf==2.0.6

print("🔄 3/5 Fairseq kutubxonasi moslashtirilib o'rnatilmoqda...")
WORKDIR = "/kaggle/working"
FAIRSEQ_DIR = os.path.join(WORKDIR, "fairseq")
if not os.path.exists(FAIRSEQ_DIR):
    !git clone --depth 1 https://github.com/pytorch/fairseq.git {FAIRSEQ_DIR}

# Fairseq OmegaConf va Python moslik patchlari
for fpath in [os.path.join(FAIRSEQ_DIR, "fairseq/__init__.py"), os.path.join(FAIRSEQ_DIR, "fairseq/dataclass/initialize.py")]:
    if os.path.exists(fpath):
        with open(fpath, 'r', encoding='utf-8') as f: c = f.read()
        c = c.replace('hydra_init()', '# hydra_init()')
        c = re.sub(r'def hydra_init\([^)]*\)[^:]*:', 'def hydra_init(*args, **kwargs):\n    return', c)
        with open(fpath, 'w', encoding='utf-8') as f: f.write(c)

!pip install -q --no-warn-conflicts --no-build-isolation --editable {FAIRSEQ_DIR}

print("🔄 4/5 RVC audio, ONNX, yt-dlp va Gradio kutubxonalari o'rnatilmoqda...")
!pip install -q --no-warn-conflicts pedalboard pydub pyworld praat-parselmouth faiss-cpu yt-dlp torchcrepe sox soundfile librosa deemix tqdm pyngrok onnxruntime-gpu gradio-client==0.8.1 "gradio==3.50.2" "fastapi<0.110" "starlette<0.37"

# Gradio frpc binar fayli (Gradio share ishlashi uchun)
try:
    import gradio, urllib.request
    gradio_dir = os.path.dirname(gradio.__file__)
    frpc_path = os.path.join(gradio_dir, "frpc_linux_amd64_v0.2")
    if not os.path.exists(frpc_path):
        urllib.request.urlretrieve("https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_amd64", frpc_path)
        os.chmod(frpc_path, 0o755)
except Exception as e:
    pass

print("🔄 5/5 AICoverGen repozitoriysi va tayyor AI modellar yuklanmoqda...")
REPO_DIR = os.path.join(WORKDIR, "HRVC")
if not os.path.exists(os.path.join(REPO_DIR, "src/webui.py")):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    !git clone https://github.com/nidayboy/aicovergenN.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin main && git reset --hard origin/main

%cd {REPO_DIR}
!python src/download_models.py

clear_output()
print("="*65)
print("✅ TABRIKLAYMIZ! BARCHA KUTUBXONA VA MODELLAR TAYYOR BO'LDI!")
print("👉 Endi pastdagi 2-QADAM orqali WebUI ni ishga tushiring.")
print("="*65)


In [ ]:
# ==========================================================
# 🚀 2-QADAM: AICoverGen WebUI ni ishga tushirish
# ==========================================================
import os, sys, subprocess, re, time
from IPython.display import HTML, display

# Sozlamalar:
PORT = 7860
USE_CLOUDFLARE = True   # Bepul, tokensiz, eng barqaror havola (Tavsiya etiladi)
USE_GRADIO = True       # Gradio.live havolasi
USE_NGROK = False       # Agar sizda Ngrok authtoken bo'lsa True qiling
NGROK_TOKEN = ""        # https://dashboard.ngrok.com/get-started/your-authtoken

REPO_DIR = "/kaggle/working/HRVC"
%cd {REPO_DIR}

# Eski qolib ketgan jarayonlarni tozalash
!fuser -k 7860/tcp 2>/dev/null || true
!pkill -9 -f "webui.py" 2>/dev/null || true
!pkill -9 -f "cloudflared" 2>/dev/null || true

os.environ["TENSORBOARD_NO_TF"] = "1"

cloudflared_url = None
if USE_CLOUDFLARE:
    cf_log = "/kaggle/working/cloudflared.log"
    if os.path.exists(cf_log): os.remove(cf_log)
    cf_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    print("⏳ Cloudflare tunnel ishga tushmoqda...")
    for _ in range(30):
        time.sleep(1)
        line = cf_proc.stdout.readline()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            cloudflared_url = match.group(0)
            break

ngrok_url = None
if USE_NGROK and NGROK_TOKEN.strip():
    try:
        from pyngrok import ngrok
        ngrok.set_auth_token(NGROK_TOKEN.strip())
        ngrok.kill()
        tunnel = ngrok.connect(PORT)
        ngrok_url = tunnel.public_url
    except Exception as e:
        print(f"⚠️ Ngrok xatosi: {e}")

cmd = ["python", "-u", "src/webui.py", "--listen", "--listen-port", str(PORT)]
if USE_GRADIO:
    cmd.append("--share")

print("\n" + "="*60)
print("⏳ WebUI serveri yuklanmoqda...")
print("="*60 + "\n")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

gradio_url = None
banner_shown = False

def show_banner(cf_link, gr_link, ng_link):
    buttons = ""
    if cf_link:
        buttons += f'''<a href="{cf_link}" target="_blank" style="background: #f38020; color: white; padding: 12px 22px; text-decoration: none; border-radius: 8px; font-weight: bold; font-size: 15px; margin: 6px; display: inline-block;">⚡ CLOUDFLARE SILKASI (Tavsiya)</a>'''
    if gr_link:
        buttons += f'''<a href="{gr_link}" target="_blank" style="background: #ff5722; color: white; padding: 12px 22px; text-decoration: none; border-radius: 8px; font-weight: bold; font-size: 15px; margin: 6px; display: inline-block;">🌐 GRADIO SILKASI</a>'''
    if ng_link:
        buttons += f'''<a href="{ng_link}" target="_blank" style="background: #2196f3; color: white; padding: 12px 22px; text-decoration: none; border-radius: 8px; font-weight: bold; font-size: 15px; margin: 6px; display: inline-block;">🚀 NGROK SILKASI</a>'''

    html = f'''
    <div style="background: #1e1e2f; border: 2px solid #6366f1; border-radius: 12px; padding: 20px; text-align: center; margin: 15px 0;">
        <h2 style="color: #fff; margin: 0 0 10px 0;">🎉 AICoverGen WebUI Tayyor!</h2>
        <p style="color: #cbd5e1; font-size: 14px;">Dastur interfeysini ochish uchun quyidagi tugmalardan birini bosing:</p>
        <div style="margin: 15px 0;">{buttons}</div>
        <div style="color: #94a3b8; font-size: 13px; text-align: left; max-width: 600px; margin: 0 auto;">
            {f'<p>• <b>Cloudflare:</b> <a href="{cf_link}" target="_blank" style="color: #fb923c;">{cf_link}</a></p>' if cf_link else ''}
            {f'<p>• <b>Gradio:</b> <a href="{gr_link}" target="_blank" style="color: #ff7043;">{gr_link}</a></p>' if gr_link else ''}
            {f'<p>• <b>Ngrok:</b> <a href="{ng_link}" target="_blank" style="color: #60a5fa;">{ng_link}</a></p>' if ng_link else ''}
        </div>
    </div>
    '''
    display(HTML(html))

if cloudflared_url:
    show_banner(cloudflared_url, None, ngrok_url)
    banner_shown = True

for line in iter(process.stdout.readline, ''):
    print(line, end='', flush=True)
    if 'Running on public URL:' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', line)
        if match:
            gradio_url = match.group(0)
            if not banner_shown or not cloudflared_url:
                show_banner(cloudflared_url, gradio_url, ngrok_url)
                banner_shown = True
